In [1]:
%load_ext autoreload
%autoreload 2
from shared_modules.data_module import DataModule
from shared_modules.utils import load_config
from shared_modules.plotting import slice_comparison_multi
from shared_modules.xai import normalize_and_clamp, channel_wise_normalize_and_clamp#, compute_ablation_cam_3d, compute_ablation_cam_3d_direct

from trainer import LitModel
import torch
from monai.transforms.intensity.array import ScaleIntensityRangePercentiles
from captum.attr import Occlusion
from captum.attr import Saliency
from pathlib import Path

<frozen importlib._bootstrap_external>:1328: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


If you have questions or suggestions, feel free to open an issue at https://github.com/DIAGNijmegen/picai_eval



Please cite the following paper when using Report Guided Annotations:

Bosma, J.S., et al. "Semi-supervised learning with report-guided lesion annotation for deep learning-based prostate cancer detection in bpMRI" to be submitted


If you have questions or suggestions, feel free to open an issue at https://github.com/DIAGNijmegen/Report-Guided-Annotation



In [19]:
# Output folder for saving images
OUTPUT_DIR = Path("../../../xai_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


# Settings:
SAVE_PREDS=False
SAVE_PROB_MAPS=False
dataset="picai" 
label_key = "pca"
config = load_config("config.yaml")
gpu = 0
fold = 0
config.gpus = [gpu]
config.cache_rate = 0.0
config.transforms.label_keys = ["pca", "prostate_pred", "zones"]
checkpoint_path = f"/cluster/home/bragehk/U-MambaMTL-XAI/gc_algorithms/base_container/models/umamba_mtl/weights/f{fold}.ckpt"
config.data.json_datalist = f"../../../json_datalists/picai/fold_{fold}.json"
model = LitModel.load_from_checkpoint(checkpoint_path, config=config)

model = model.eval()
model.to(gpu)

LitModel(
  (model): UMambaBot(
    (encoder): UNetResEncoder(
      (stem): Sequential(
        (0): BasicResBlock(
          (conv1): Conv3d(3, 32, kernel_size=(1, 3, 3), stride=(1, 1, 1), padding=(0, 1, 1))
          (norm1): InstanceNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
          (act1): LeakyReLU(negative_slope=0.01, inplace=True)
          (conv2): Conv3d(32, 32, kernel_size=(1, 3, 3), stride=(1, 1, 1), padding=(0, 1, 1))
          (norm2): InstanceNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
          (act2): LeakyReLU(negative_slope=0.01, inplace=True)
          (conv3): Conv3d(3, 32, kernel_size=(1, 1, 1), stride=(1, 1, 1))
        )
        (1): BasicBlockD(
          (conv1): ConvDropoutNormReLU(
            (conv): Conv3d(32, 32, kernel_size=(1, 3, 3), stride=(1, 1, 1), padding=(0, 1, 1))
            (norm): InstanceNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
            (no

# Load existing explainability maps

In [1]:
from monai.utils.misc import first
# Load a occlusion and saliency map
occlusion_map = torch.load(f"../../../xai_outputs/umamba_mtl/f0/sample_074_10392_1000398_0000.nii.gz/occlusion_map.pt", weights_only=False)
attention_map = torch.load(f"../../../xai_outputs/umamba_mtl/f0/sample_074_10392_1000398_0000.nii.gz/saliency_map.pt", weights_only=False)

dm = DataModule(config, debug_index=74)
dm.setup("debug")
dl = dm.debug_dataloader()

batch = first(dl)

case_id = batch["image"].meta["filename_or_obj"][0].split("/")[-1]
print(f"Case ID: {case_id}")

logits = model(batch["image"].to(gpu))

<frozen importlib._bootstrap_external>:1328: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


NameError: name 'torch' is not defined

# Generate your own saliency and occlusion map

In [55]:
from monai.utils.misc import first

for case_id in range(65, 100):
    dm = DataModule(
        config=config,
        debug_index=case_id
    )
    dm.setup("debug")
    dl = dm.debug_dataloader()

    batch = first(dl)
    if batch["pca"].max() == 0:
        print("Not any PCa here!")
        continue


    logits = model(batch["image"].to(gpu))
    patient_id = batch["image"].meta["filename_or_obj"][0].split("/")[-1]
    print(f"Patient id: {patient_id}")
    confidence = round(torch.sigmoid(logits)[0,1].max().item() * 100, 2)
    print(f"PCa confidence: {confidence}%")
    if confidence > 55:
        print("Case id ", case_id)
        break


# Load a occlusion and saliency map
#occlusion_map = torch.load(OUTPUT_DIR / f"umamba_mtl/f1/sample_00{case_id}_{patient_id}/occlusion_map.pt", weights_only=False)[0]
#attention_map = torch.load(OUTPUT_DIR / f"umamba_mtl/f1/sample_00{case_id}_{patient_id}/saliency_map.pt", weights_only=False)[0]

Not any PCa here!
Patient id: 10268_1000272_0000.nii.gz
PCa confidence: 86.48%
Case id  66


In [27]:
def agg_segmentation_wrapper(inp):
    model_out = model(inp)[:, 0:2, ...]  # (B, 2, H, W, D)
    pca_mask = (model_out[:, 1, ...].sigmoid() > 0.5).float()  # (B, H, W, D)
    aggregated_logits = (model_out[:, 1, ...] * pca_mask).sum(dim=(1, 2, 3))  # (B,)
    return aggregated_logits


r = torch.randn((3, 3, 128, 128, 20), device="cuda:0")

print(agg_segmentation_wrapper(r))

occlusion = Occlusion(agg_segmentation_wrapper)
attribute_fn = Saliency(agg_segmentation_wrapper)

size = 16
sliding_window_shapes = (1, size, size, 2) 

size_stride = 8
strides = (1, size_stride, size_stride, 1)               

perturbations_per_eval = 1

tensor([ 52.4340,  24.4967, 612.1533], device='cuda:0', grad_fn=<SumBackward1>)


In [ ]:
import time

batch = next(iter(dl))

x = batch["image"].to(gpu)
x = x.as_tensor() if hasattr(x, 'as_tensor') else x

baselines = 0 #x.mean(dim=(2,3,4), keepdim=True)
print(f"baselines: {baselines}")


logits = model(x)
print(logits.shape)


print("max logit", logits.max())

target = 1

t0 = time.time()
attention_map = attribute_fn.attribute(x, target=target, abs=True)
print(f"Saliency took {time.time() - t0:.2f}s")
print(f"saliency max: ", attention_map.max())
t0 = time.time()
occlusion_map: torch.Tensor = occlusion.attribute(
    x,
    sliding_window_shapes=sliding_window_shapes,
    strides=strides,
    baselines=baselines,
    target=target,
    perturbations_per_eval=perturbations_per_eval,
    show_progress=True
)
print(f"occlusion_map max: ", occlusion_map.max())
print(f"Occlusion took {time.time() - t0:.2f}s")
patient_id = batch["image"].meta["filename_or_obj"][0].split("/")[-1]
print(f"patient id: {patient_id}")

baselines: 0
torch.Size([1, 5, 128, 128, 20])
max logit tensor(14.7705, device='cuda:0', grad_fn=<MaxBackward1>)
Saliency took 0.09s
saliency max:  tensor(0., device='cuda:0')


Occlusion attribution:   0%|          | 0/983041 [00:00<?, ?it/s]

occlusion_map max:  tensor(0., device='cuda:0')
Occlusion took 125.15s
patient id: 10268_1000272_0000.nii.gz


# Interactive explanation viewer

In [ ]:
CHANNEL_WISE_NORM = False
print(f"logit shape {logits.shape}")
print(f"logit max {logits[:, 1].max()}")
print(f"logit min {logits[:, 1].min()}")
print(f"logit max {logits[:, 0].max()}")
print(f"logit min {logits[:, 0].min()}")

img = batch["image"][0]
gt = batch["pca"][0]
pred = (torch.sigmoid(logits) >= 0.2).int()[0][1][None].to("cpu").float()

print(f"Pred sum: {pred.sum()}, gt sum: {gt.sum()}")
print(f"{torch.where(pred == 1.0)}")

logit = logits[0][1][None].to("cpu")

print(f"gt shape: {gt.shape}, max: {gt.max()}, min: {gt.min()}")
print(f"pred shape: {pred.shape}, max: {pred.max()}, min: {pred.min()}")

acc = attention_map[0].to("cpu")
occ = occlusion_map[0].to("cpu")

for channel in range(occ.shape[0]):
    print(f"Channel: {channel}")
    print(f"occ max: {occ[channel].max()}")
    print(f"occ min: {occ[channel].min()}")

print(f"acc shape: {acc.shape}, max: {acc.max()}, min: {acc.min()}")
print(f"occ shape: {occ.shape}, max: {occ.max()}, min: {occ.min()}")


acc = channel_wise_normalize_and_clamp(acc) if CHANNEL_WISE_NORM else normalize_and_clamp(acc)
occ = channel_wise_normalize_and_clamp(occ) if CHANNEL_WISE_NORM else normalize_and_clamp(occ)

#print(f"More pca in pz: {pca_in_pz > pca_in_tz}") 
#print(f"pca in tz: {pca_in_tz}") 
#print(f"pca in pz: {pca_in_pz}")
case_id = batch["image"].meta["filename_or_obj"][0].split("/")[-1]
print(f"Case ID: {case_id}")
confidence = round(torch.sigmoid(logits)[0,1].max().item() * 100, 2)
print(f"PCa confidence: {confidence}%")

slice_comparison_multi(image=ScaleIntensityRangePercentiles(lower=0, upper=100, b_min=0, b_max=1)(img), labels=[gt, pred, acc, occ], titles=["original", "Ground Truth","Prediction", "Saliency", "Occlusion"])

logit shape torch.Size([1, 5, 128, 128, 20])
logit max 0.042187318205833435
logit min -10.678226470947266
logit max 8.436287879943848
logit min -0.8011983036994934
Pred sum: 1163.0, gt sum: 5620.0
(metatensor([0, 0, 0,  ..., 0, 0, 0]), metatensor([32, 32, 32,  ..., 61, 61, 63]), metatensor([64, 65, 66,  ..., 90, 91, 89]), metatensor([8, 8, 8,  ..., 8, 8, 8]))
gt shape: torch.Size([1, 128, 128, 20]), max: 1.0, min: 0.0
pred shape: torch.Size([1, 128, 128, 20]), max: 1.0, min: 0.0
Channel: 0
occ max: 0.04371051490306854
occ min: -6.42797327041626
Channel: 1
occ max: 0.04371051490306854
occ min: -0.03358304500579834
Channel: 2
occ max: 0.04371051490306854
occ min: -0.06061451509594917
acc shape: torch.Size([3, 128, 128, 20]), max: 0.15639285743236542, min: 1.4279066817834973e-10
occ shape: torch.Size([3, 128, 128, 20]), max: 0.04371051490306854, min: -6.42797327041626
Max val:
metatensor(0.1564)
Max val:
metatensor(0.0437)
Case ID: 10032_1000032_0000.nii.gz
PCa confidence: 51.05%


interactive(children=(IntSlider(value=0, description='Slice Index:', max=19), Output()), _dom_classes=('widget…

In [51]:
from shared_modules.plotting import slice_comparison_multi_gif

slice_comparison_multi_gif(image=ScaleIntensityRangePercentiles(lower=0, upper=100, b_min=0, b_max=1)(img), labels=[gt, pred, acc, occ], titles=["original", "Ground Truth","Prediction", "Saliency", "Occlusion"])

Saved pendulum GIF (38 frames) to slice_comparison.gif


'slice_comparison.gif'

In [ ]:
# Then your function call
cam_volume = compute_ablation_cam_3d_direct(
    model=model,
    input_tensor=x,
    target_category=1,
    target_mask=pred[0].cpu().numpy(),
    device=gpu,
    ablation_size=(16, 16, 5),
    stride=(16, 16, 1),
)


AblationCAM 3D: 100%|██████████| 16/16 [05:18<00:00, 19.91s/it]


In [29]:
cam_volume.shape

(256, 256, 20)

In [40]:
img.shape

torch.Size([3, 256, 256, 20])

In [ ]:
from ipywidgets import interact, IntSlider
import matplotlib.pyplot as plt

def view_volume(volume, title="Volume", cmap="viridis", vmin=None, vmax=None):
    """Interactive viewer for 3D volume with slider."""
    if vmin is None:
        vmin = volume.min()
    if vmax is None:
        vmax = volume.max()
    
    @interact(slice_idx=IntSlider(min=0, max=volume.shape[-1]-1, value=volume.shape[-1]//2, description='Slice:'))
    def compare_slices(slice_idx):
        fig, axes = plt.subplots(1, 4, figsize=(15, 5))
    
        # Original image (e.g., T2W channel)
        axes[0].imshow(img[0, :, :, slice_idx].cpu(), cmap='gray')
        axes[0].set_title('T2W')
    
        # Ground truth
        axes[1].imshow(gt[0, :, :, slice_idx].cpu(), cmap='Reds', vmin=0, vmax=1)
        axes[1].set_title('Ground Truth')
        
        # Prediction
        axes[2].imshow(pred[0, :, :, slice_idx].cpu(), cmap='Reds', vmin=0, vmax=1)
        axes[2].set_title('Prediction')
    
        # AblationCAM
        im = axes[3].imshow(cam_volume[:, :, slice_idx], cmap='hot', vmin=vmin, vmax=vmax)
        axes[3].set_title('AblationCAM')
        plt.colorbar(im, ax=axes[3])
    
        for ax in axes:
            ax.axis('off')
        plt.tight_layout()
        plt.show()


view_volume(cam_volume, title="AblationCAM", cmap="hot")


interactive(children=(IntSlider(value=10, description='Slice:', max=19), Output()), _dom_classes=('widget-inte…

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import io

def create_volume_gif(img, gt, pred, cam_volume, output_path="volume.gif", fps=5, dpi=100, ping_pong=False):
    """
    Create a GIF animation of the volume slices.
    
    Parameters:
    -----------
    img : tensor
        Input image tensor (channels, H, W, D)
    gt : tensor
        Ground truth tensor (channels, H, W, D)
    pred : tensor
        Prediction tensor (channels, H, W, D), values 0-1
    cam_volume : array
        CAM volume array (H, W, D)
    output_path : str
        Output path for the GIF file
    fps : int
        Frames per second for the GIF
    dpi : int
        Resolution of each frame
    ping_pong : bool
        If True, play forward then backward for smooth looping
    """
    # Get global min/max for consistent color scaling
    vmin = cam_volume.min()
    vmax = cam_volume.max()
    
    num_slices = cam_volume.shape[-1]
    frames = []
    
    for slice_idx in range(num_slices):
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        
        # Original image (e.g., T2W channel)
        axes[0].imshow(img[0, :, :, slice_idx].cpu(), cmap='gray')
        axes[0].set_title('T2W')
        
        # Ground truth
        axes[1].imshow(gt[0, :, :, slice_idx].cpu(), cmap='Reds', vmin=0, vmax=1)
        axes[1].set_title('Ground Truth')
        
        # Prediction
        axes[2].imshow(pred[0, :, :, slice_idx].cpu(), cmap='Reds', vmin=0, vmax=1)
        axes[2].set_title('Prediction')
        
        # AblationCAM
        im = axes[3].imshow(cam_volume[:, :, slice_idx], cmap='hot', vmin=vmin, vmax=vmax)
        axes[3].set_title('AblationCAM')
        plt.colorbar(im, ax=axes[3])
        
        for ax in axes:
            ax.axis('off')
        
        # Add slice indicator
        fig.suptitle(f'Slice {slice_idx + 1}/{num_slices}', fontsize=12)
        plt.tight_layout()
        
        # Convert figure to PIL Image
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight')
        buf.seek(0)
        frame = Image.open(buf).convert('RGB')
        frames.append(frame.copy())
        buf.close()
        plt.close(fig)
    
    # Add reverse frames for ping-pong effect
    if ping_pong:
        frames = frames + frames[-2:0:-1]
    
    # Save as GIF
    duration = int(1000 / fps)  # Duration per frame in milliseconds
    frames[0].save(
        output_path,
        save_all=True,
        append_images=frames[1:],
        duration=duration,
        loop=0  # 0 means infinite loop
    )
    
    print(f"GIF saved to {output_path} ({len(frames)} frames)")
    return output_path

In [46]:
create_volume_gif(img, gt, pred, cam_volume, output_path="ablation_cam.gif", fps=5, ping_pong=True)

GIF saved to ablation_cam.gif (38 frames)


'ablation_cam.gif'

In [41]:
cam_volume_2 = compute_ablation_cam_3d(
    model=model,
    input_tensor=x,
    target_category=1,  # PCa channel
    target_mask=pred[0].cpu().numpy(),  # Focus on predicted region
    device=gpu
)

Computing AblationCAM per slice:  20%|██        | 4/20 [00:01<00:05,  2.78it/s]

Computing AblationCAM per slice:  30%|███       | 6/20 [00:01<00:03,  3.62it/s]

Computing AblationCAM per slice: 100%|██████████| 20/20 [00:02<00:00,  9.41it/s]

made  20  slices


In [ ]:
view_volume(cam_volume_2)

interactive(children=(IntSlider(value=10, description='Slice:', max=19), Output()), _dom_classes=('widget-inte…

In [67]:
from shared_modules.xai import compute_xai_metrics_segmentation

metrics = compute_xai_metrics_segmentation(
    model=model,
    inputs=x,
    attributions=attention_map,
    target=1,  # PCa class
    aggregation="predicted",  # or "masked" with mask=your_mask
    n_perturb_samples=100,
    agg_wrapper=agg_segmentation_wrapper,
    perturb_radius=0.2
)

print(metrics)

{'infidelity': np.float64(19.025664772666417), 'sensitivity': 0.4467165797449239}


{'infidelity': 109.40544798374177, 'sensitivity': np.float64(0.009333162949272234)}
